This script is a **stable runner to test a classification pipeline** on the crime dataset, and it is written to avoid common Windows problems.

**Main idea:**
1. It first checks that an API key exists, because AIDE needs it to generate and improve code.
2. It finds `crime_data.csv/xlsx`(kaggle) by searching under `C:\TUB\RDEP`, so the run does not depend on a hardcoded file path.
3. It creates a clean input folder (`AIDE_CRIME_INPUT`) and copies the dataset there.  
   This makes the run reproducible and avoids mixing old files.
4. It writes a task description that forces a **single classification task**:
   - automatically choose a target column,
   - use a standard scikit-learn pipeline (OneHotEncoder + StandardScaler + LogisticRegression),
   - evaluate with **ACC and F1 only** (no regression metrics),
   - save `predictions.csv` and `model.joblib`.
5. It calls AIDE with a **fast configuration** (`agent.steps=5`, no CV, no report) and uses `copy_data=True` to avoid Windows symlink errors.

In short, this script makes the evaluation reliable by keeping the input clean, forcing consistent classification metrics, and saving logs and outputs for later checking.



In [2]:
# run_aide_crime_local.py
# Fast & stable AIDE runner for a SINGLE classification task on crime data (Windows-safe)

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path


def find_data_file(start: Path) -> Path:
    """
    Recursively search for crime_data.csv / crime_data.xlsx / crime_data.xls.
    """
    for ext in ("*.csv", "*.xlsx", "*.xls"):
        for p in start.rglob(ext):
            if p.stem.lower() == "crime_data":
                return p
    raise FileNotFoundError(f"Cannot find crime_data.csv/xlsx/xls under {start}")


def run_cmd_stream(cmd: list[str], log_path: Path) -> None:
    """
    Run a command, stream output in real time, and save full output to a log file.
    Uses bytes mode + UTF-8 decode with replacement to avoid Windows encoding crashes.
    """
    print("\n[CMD]")
    print(" ".join(cmd))
    print(f"[LOG FILE] {log_path}")

    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")

    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("w", encoding="utf-8", errors="replace") as f:
        p = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=False,  # bytes mode
            env=env,
        )
        assert p.stdout is not None

        for chunk in iter(lambda: p.stdout.readline(), b""):
            line = chunk.decode("utf-8", errors="replace")
            print(line, end="")
            f.write(line)

        ret = p.wait()
        f.write(f"\n[EXIT_CODE] {ret}\n")

    if ret != 0:
        raise RuntimeError(
            f"AIDE failed with exit code {ret}. See log for details: {log_path}"
        )


def main() -> None:
    # ---- API key sanity check ----
    if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY")):
        raise RuntimeError("No API key found. Set OPENAI_API_KEY (or similar).")

    # ---- Locate dataset ----
    project_root = Path(r"C:\TUB\RDEP")
    data_file = find_data_file(project_root)
    print(f"Found dataset: {data_file}")

    # ---- Prepare AIDE directories ----
    base_dir = Path.home() / "Documents"
    data_dir = base_dir / "AIDE_CRIME_INPUT"
    ws_dir = base_dir / "AIDE_CRIME_WORKSPACE"

    data_dir.mkdir(parents=True, exist_ok=True)
    ws_dir.mkdir(parents=True, exist_ok=True)

    # Clean input dir
    for p in data_dir.glob("*"):
        if p.is_file():
            p.unlink()

    shutil.copy2(data_file, data_dir / data_file.name)

    print(f"\nData dir: {data_dir}")
    print(f"Workspace: {ws_dir}")
    print("Files in input dir:")
    for p in data_dir.iterdir():
        print(" -", p.name)

    # ---- Task description (CLASSIFICATION ONLY) ----
    desc = """
Generate a SINGLE runnable Python script named solution.py.

Data:
- A file named crime_data.csv OR crime_data.xlsx will be present under the working directory.
- The script MUST locate it by recursively searching from the current directory.
- If the file is Excel, load it with pandas.read_excel; if CSV, use pandas.read_csv.

Task:
- Load the dataset into a pandas DataFrame.
- Determine the target column:
  1) if any of these columns exist, use the first found: target, y, label, value
  2) otherwise use the last column
- Define X as all remaining columns, y as the target.

IMPORTANT: SINGLE TASK ONLY
- This is a CLASSIFICATION task ONLY.
- Do NOT attempt regression.
- Use classification metrics only (ACC, F1).
- Ensure all candidate solutions use the same metric direction (maximize only).

Modeling:
- Build a scikit-learn Pipeline:
  - Categorical: OneHotEncoder(handle_unknown="ignore")
  - Numeric: StandardScaler
  - ColumnTransformer
- Model: LogisticRegression(max_iter=2000)
- Split: train_test_split(test_size=0.2, random_state=42)

Outputs:
- Print: VAL_ACC=<float>, VAL_F1=<float>
- Save predictions.csv with y_true, y_pred, and features.

Prediction API:
- Implement predict_csv(input_path, output_path)
- Save trained pipeline to model.joblib

Constraints:
- Use numpy, pandas, scikit-learn, joblib only.
- English comments only.
""".strip()

    desc_path = ws_dir / "desc.txt"
    desc_path.write_text(desc, encoding="utf-8")
    print(f"\nWrote description to: {desc_path}")

    # ---- Call AIDE (FAST CONFIG; Windows-safe) ----
    cmd = [
        "aide",
        f"data_dir={str(data_dir)}",
        f"workspace_dir={str(ws_dir)}",
        f"desc_file={str(desc_path)}",

        # SPEED OPTIMIZATION
        "agent.steps=5",               # default ~20
        "agent.k_fold_validation=1",   # disable CV
        "generate_report=False",       # skip final LLM report

        # IMPORTANT: Windows symlink privilege error fix
        # copy_data=False => tries symlinks => WinError 1314
        "copy_data=True",              # force real copy
    ]

    log_file = ws_dir / "aide_last_run.log"
    print("\nCalling AIDE (fast mode, no symlinks)...")
    run_cmd_stream(cmd, log_file)

    print("\nDone.")
    print("Workspace:", ws_dir)
    print("Check the newest experiment subfolder for solution.py.")


if __name__ == "__main__":
    main()


Found dataset: C:\TUB\RDEP\crime_data.csv

Data dir: C:\Users\wenyi\Documents\AIDE_CRIME_INPUT
Workspace: C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE
Files in input dir:
 - crime_data.csv

Wrote description to: C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\desc.txt

Calling AIDE (fast mode, no symlinks)...

[CMD]
aide data_dir=C:\Users\wenyi\Documents\AIDE_CRIME_INPUT workspace_dir=C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE desc_file=C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\desc.txt agent.steps=5 agent.k_fold_validation=1 generate_report=False copy_data=True
[LOG FILE] C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\aide_last_run.log
⠋ Preparing agent workspace (copying and extracting files) ...
┌───────── AIDE is working on experiment: "2-ambrosial-curvy-beaver" ─────────┐
│                                                                             │
│  Result visualization:                                                      │
│  ▶ C:\Users\wenyi\AppData\Local\Programs\Micr

## Fairness Testing Idea 

This script runs AIDE on a **crime dataset** to test a **multiclass classification model** with a focus on **fairness**.

**Main idea:**  
The goal is not only high accuracy, but also to reduce unfair differences between groups (for example, different genders).

**How fairness is tested:**
1. The dataset is loaded from a local copy (downloaded from Kaggle).
2. A target column is selected automatically, and the task is treated as multiclass classification.
3. If a sensitive column exists (such as `Offender_Gender` or `Victim_Gender`), it is used only for evaluation, not as a model input.
4. For each class, fairness is checked in a one-vs-rest way:
   - True Positive Rate (TPR) for each group  
   - False Positive Rate (FPR) for each group  
   - Prediction rate for each group
5. The script computes gaps between groups, such as TPR gaps and prediction rate gaps.
6. These gaps are combined into a fairness penalty.
7. A single final score is used:
VAL_SCORE = Macro F1 − fairness penalty
A higher score means better accuracy and better fairness.

**Why this matters:**  
This approach shows whether the model treats different groups in a similar way, without using sensitive attributes for prediction. It gives one clear score to compare models.

**References:**  
- Hardt et al. (2016). *Equality of Opportunity in Supervised Learning*.  
- Barocas & Selbst (2016). *Big Data’s Disparate Impact*.  
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*.


In [13]:
# run_aide_crime_local.py
# Fast & stable AIDE runner for a SINGLE classification task on crime data (Windows-safe)

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path


def find_data_file(start: Path) -> Path:
    """
    Recursively search for crime_data.csv / crime_data.xlsx / crime_data.xls.
    """
    for ext in ("*.csv", "*.xlsx", "*.xls"):
        for p in start.rglob(ext):
            if p.stem.lower() == "crime_data":
                return p
    raise FileNotFoundError(f"Cannot find crime_data.csv/xlsx/xls under {start}")


def run_cmd_stream(cmd: list[str], log_path: Path) -> None:
    """
    Run a command, stream output in real time, and save full output to a log file.
    Uses bytes mode + UTF-8 decode with replacement to avoid Windows encoding crashes.
    """
    print("\n[CMD]")
    print(" ".join(cmd))
    print(f"[LOG FILE] {log_path}")

    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")

    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("w", encoding="utf-8", errors="replace") as f:
        p = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=False,  # bytes mode
            env=env,
        )
        assert p.stdout is not None

        for chunk in iter(lambda: p.stdout.readline(), b""):
            line = chunk.decode("utf-8", errors="replace")
            print(line, end="")
            f.write(line)

        ret = p.wait()
        f.write(f"\n[EXIT_CODE] {ret}\n")

    if ret != 0:
        raise RuntimeError(
            f"AIDE failed with exit code {ret}. See log for details: {log_path}"
        )


def main() -> None:
    # ---- API key sanity check ----
    if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY")):
        raise RuntimeError("No API key found. Set OPENAI_API_KEY (or similar).")

    # ---- Locate dataset ----
    project_root = Path(r"C:\TUB\RDEP")
    data_file = find_data_file(project_root)
    print(f"Found dataset: {data_file}")

    # ---- Prepare AIDE directories ----
    base_dir = Path.home() / "Documents"
    data_dir = base_dir / "AIDE_CRIME_INPUT"
    ws_dir = base_dir / "AIDE_CRIME_WORKSPACE"

    data_dir.mkdir(parents=True, exist_ok=True)
    ws_dir.mkdir(parents=True, exist_ok=True)

    # Clean input dir
    for p in data_dir.glob("*"):
        if p.is_file():
            p.unlink()

    shutil.copy2(data_file, data_dir / data_file.name)

    print(f"\nData dir: {data_dir}")
    print(f"Workspace: {ws_dir}")
    print("Files in input dir:")
    for p in data_dir.iterdir():
        print(" -", p.name)

    # ---- Task description (CLASSIFICATION ONLY, FAIRNESS-AWARE) ----
    desc = """
Generate a SINGLE runnable Python script named solution_fair.py.

Data:
- A file named crime_data.csv OR crime_data.xlsx will be present under the working directory.
- The script MUST locate it by recursively searching from the current directory.
- If the file is Excel, load it with pandas.read_excel; if CSV, use pandas.read_csv.

Task:
- Load the dataset into a pandas DataFrame.
- Determine the target column:
  1) if any of these columns exist, use the first found: target, y, label, value
  2) otherwise use the last column
- Define X as all remaining columns, y as the target.
- Treat this as a MULTICLASS CLASSIFICATION task (do not binarize y).

Fairness requirement:
- Compute group fairness by a sensitive attribute column if it exists:
  - Preferred: Offender_Gender
  - If not present: Victim_Gender
  - If neither present: skip fairness computation but still run the model.
- Fairness evaluation must be multiclass one-vs-rest per class:
  For each class c:
    - y_true_pos = (y_true == c)
    - y_pred_pos = (y_pred == c)
    - compute per-group TPR (recall), FPR, and predicted rate for class c
  Then compute per-class gaps across groups:
    - gap_tpr_c = max(TPR_g) - min(TPR_g)
    - gap_predrate_c = max(P(yhat=c|g)) - min(P(yhat=c|g))
- Focus fairness penalty on "major" classes (support >= 50) to avoid tiny-class noise.

Modeling:
- Build a scikit-learn Pipeline:
  - Categorical: OneHotEncoder(handle_unknown="ignore")
  - Numeric: StandardScaler
  - ColumnTransformer
- Use LogisticRegression with class balancing to reduce bias from imbalance:
  LogisticRegression(max_iter=4000, class_weight="balanced", n_jobs=None)
- Split: train_test_split(test_size=0.2, random_state=42, stratify=y)

Evaluation (single metric to maximize):
- Compute validation macro-F1: VAL_F1_MACRO
- Compute fairness penalty:
  - FAIR_TPR_GAP = average(gap_tpr_c over major classes)
  - FAIR_PREDRATE_GAP = average(gap_predrate_c over major classes)
  - FAIR_PENALTY = 0.5*FAIR_TPR_GAP + 0.5*FAIR_PREDRATE_GAP
- Define a single objective score to MAXIMIZE:
  VAL_SCORE = VAL_F1_MACRO - 0.3 * FAIR_PENALTY
- Print exactly:
  VAL_SCORE=<float>
  VAL_F1_MACRO=<float>
  FAIR_TPR_GAP=<float>
  FAIR_PREDRATE_GAP=<float>
  FAIR_PENALTY=<float>
- Also print a small table-like summary per group for the top 3 most frequent classes.

Outputs:
- Save model to model.joblib (joblib.dump).
- Save predictions.csv containing:
  - y_true
  - y_pred
  - (if available) the chosen sensitive column (e.g., Offender_Gender)
  - plus all feature columns.

Prediction API:
- Implement predict_csv(input_path, output_path)
  - Loads model.joblib
  - Predicts y_pred for rows
  - Writes output CSV with predictions.

Constraints:
- Use numpy, pandas, scikit-learn, joblib only.
- English comments only.
""".strip()

    desc_path = ws_dir / "desc.txt"
    desc_path.write_text(desc, encoding="utf-8")
    print(f"\nWrote description to: {desc_path}")

    # ---- Call AIDE (FAST CONFIG; Windows-safe) ----
    cmd = [
        "aide",
        f"data_dir={str(data_dir)}",
        f"workspace_dir={str(ws_dir)}",
        f"desc_file={str(desc_path)}",

        # SPEED OPTIMIZATION
        "agent.steps=5",               # default ~20
        "agent.k_fold_validation=1",   # disable CV
        "generate_report=False",       # skip final LLM report

        # IMPORTANT: Windows symlink privilege error fix
        "copy_data=True",              # force real copy
    ]

    log_file = ws_dir / "aide_last_run.log"
    print("\nCalling AIDE (fast mode, no symlinks)...")
    run_cmd_stream(cmd, log_file)

    print("\nDone.")
    print("Workspace:", ws_dir)
    print("Check the newest experiment subfolder for solution_fair.py.")


if __name__ == "__main__":
    main()


Found dataset: C:\TUB\RDEP\crime_data.csv

Data dir: C:\Users\wenyi\Documents\AIDE_CRIME_INPUT
Workspace: C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE
Files in input dir:
 - crime_data.csv

Wrote description to: C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\desc.txt

Calling AIDE (fast mode, no symlinks)...

[CMD]
aide data_dir=C:\Users\wenyi\Documents\AIDE_CRIME_INPUT workspace_dir=C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE desc_file=C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\desc.txt agent.steps=5 agent.k_fold_validation=1 generate_report=False copy_data=True
[LOG FILE] C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\aide_last_run.log
⠋ Preparing agent workspace (copying and extracting files) ...
┌────────── AIDE is working on experiment: "2-hot-unique-labrador" ───────────┐
│                                                                             │
│  Result visualization:                                                      │
│  ▶ C:\Users\wenyi\AppData\Local\Programs\Micr

In [31]:
from pathlib import Path

workspace = Path(r"C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE")

models = sorted(
    workspace.rglob("model.joblib"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print("Found model.joblib files (newest first):\n")
for p in models:
    print(p)


Found model.joblib files (newest first):

C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\2-adept-happy-eagle\model.joblib
C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\2-omniscient-crow-of-nirvana\model.joblib


## Fairness Testing Idea

This code compares **fairness** between two trained models (baseline vs fairness) on the **same crime dataset**.

**Main idea:**
- A model can be unfair if it predicts some classes much more often for one group than for another group.
- Here the sensitive group is `Offender_Gender`, and the target is `Category`.

**What the script does:**
1. Load the dataset and split it into:
   - `X` = all features (drop the target column)
   - `y_true` = true class labels (`Category`)
   - `g` = group labels (`Offender_Gender`)
2. Load two saved models (`model.joblib`):
   - baseline model
   - fairness-aware model
3. Make predictions with both models on the **same X**.
4. For each class, compute **predicted rate by group**:
   - `predicted_rate = P(y_hat == class | group)`
5. For each class, compute a fairness score:
   - `ratio = min(predicted_rate) / max(predicted_rate)` across groups
   - If the ratio is closer to **1**, the class is predicted more equally across groups (better fairness).
6. Compare baseline vs fairness model:
   - `delta_ratio = ratio_fair - ratio_baseline`
   - Positive delta means fairness improved for that class.

**Why this is a fairness test:**
It checks whether the model’s **output distribution** is similar across groups.  
This is related to group fairness ideas like **statistical parity / disparate impact**, but applied per class in a multiclass setting.

**References:**
- Barocas & Selbst (2016). *Big Data’s Disparate Impact*.  
- Hardt, Price, & Srebro (2016). *Equality of Opportunity in Supervised Learning*.  
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*.


In [32]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import load


# =================================================
# CONFIG (already fixed for your case)
# =================================================

DATA_PATH = Path(r"C:\TUB\RDEP\crime_data.csv")
GROUP_COL = "Offender_Gender"
TARGET_COL = "Category"

BASE_MODEL_PATH = Path(
    r"C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\2-omniscient-crow-of-nirvana\model.joblib"
)

FAIR_MODEL_PATH = Path(
    r"C:\Users\wenyi\Documents\AIDE_CRIME_WORKSPACE\2-adept-happy-eagle\model.joblib"
)


# =================================================
# FAIRNESS METRICS
# =================================================

def predicted_rates_by_group(y_pred: pd.Series, group: pd.Series, classes: list[str]) -> pd.DataFrame:
    """Predicted rate P(y_hat == class) per group."""
    rows = []
    group = group.astype(str)

    for cls in classes:
        for gval, idx in group.groupby(group).groups.items():
            idx = list(idx)
            rate = float((y_pred.iloc[idx] == cls).mean()) if len(idx) else np.nan
            rows.append({
                "class": cls,
                "group": gval,
                "n": len(idx),
                "predicted_rate": rate,
            })

    return pd.DataFrame(rows)


def disparity_ratio_min_over_max(rep: pd.DataFrame) -> pd.DataFrame:
    """Disparity ratio = min(pred_rate) / max(pred_rate) per class."""
    out = []
    for cls, sub in rep.groupby("class"):
        r = sub["predicted_rate"].dropna()
        ratio = float(r.min() / r.max()) if len(r) and r.max() > 0 else np.nan
        out.append({"class": cls, "ratio_min_over_max": ratio})
    return pd.DataFrame(out)


# =================================================
# MAIN
# =================================================

# Load data
df = pd.read_csv(DATA_PATH)

if GROUP_COL not in df.columns:
    raise ValueError(f"{GROUP_COL} not found in data.")
if TARGET_COL not in df.columns:
    raise ValueError(f"{TARGET_COL} not found in data.")

X = df.drop(columns=[TARGET_COL])
g = df[GROUP_COL].astype(str)
y_true = df[TARGET_COL].astype(str)

classes = sorted(y_true.unique().tolist())

print("Classes:", classes)
print("Group counts:", g.value_counts().to_dict())

# Load models
base_model = load(BASE_MODEL_PATH)
fair_model = load(FAIR_MODEL_PATH)

# Predict on the SAME data
pred_base = pd.Series(base_model.predict(X)).astype(str)
pred_fair = pd.Series(fair_model.predict(X)).astype(str)

# Compute fairness tables
rep_base = predicted_rates_by_group(pred_base, g, classes)
rep_fair = predicted_rates_by_group(pred_fair, g, classes)

ratio_base = (
    disparity_ratio_min_over_max(rep_base)
    .rename(columns={"ratio_min_over_max": "ratio_baseline"})
)

ratio_fair = (
    disparity_ratio_min_over_max(rep_fair)
    .rename(columns={"ratio_min_over_max": "ratio_fair"})
)

# Compare fairness
cmp = ratio_base.merge(ratio_fair, on="class", how="inner")
cmp["delta_ratio"] = cmp["ratio_fair"] - cmp["ratio_baseline"]
cmp = cmp.sort_values("delta_ratio", ascending=False)

print("\n=== Fairness comparison (predicted-rate ratio min/max) ===")
display(cmp)

print("\n=== Classes with improved fairness (delta_ratio > 0) ===")
display(cmp[cmp["delta_ratio"] > 0])

print("\n=== Classes with worsened fairness (delta_ratio < 0) ===")
display(cmp[cmp["delta_ratio"] < 0])


Classes: ['Drug and Weapon Crimes', 'Miscellaneous', 'Sexual Crimes', 'Theft', 'Vandalism', 'Violence']
Group counts: {'MALE': 4987, 'FEMALE': 1651}

=== Fairness comparison (predicted-rate ratio min/max) ===


,class,ratio_baseline,ratio_fair,delta_ratio
1,Miscellaneous,0.471511,0.742503,0.270992
3,Theft,0.597480,0.431224,-0.166256
5,Violence,0.918344,0.174660,-0.743684
0,Drug and Weapon Crimes,NaN,0.000000,NaN
2,Sexual Crimes,NaN,0.000000,NaN
4,Vandalism,NaN,0.153569,NaN



=== Classes with improved fairness (delta_ratio > 0) ===


,class,ratio_baseline,ratio_fair,delta_ratio
1,Miscellaneous,0.471511,0.742503,0.270992



=== Classes with worsened fairness (delta_ratio < 0) ===


,class,ratio_baseline,ratio_fair,delta_ratio
3,Theft,0.597480,0.431224,-0.166256
5,Violence,0.918344,0.174660,-0.743684


## Accuracy and Performance Comparison 

This code compares the **overall performance** of two models:
- a **baseline model**
- a **fairness-aware model**

**What is measured:**
1. **Accuracy**  
   - How many predictions are correct in total.
2. **Macro F1 score**  
   - The average F1 score over all classes.
   - Each class has the same weight, so rare classes also matter.

**What the code does:**
1. It computes accuracy and macro F1 for both models using the same true labels.
2. It puts the results into a table.
3. It computes the difference (`delta`) between the fairness model and the baseline:
   - `delta_accuracy` > 0 means accuracy improved.
   - `delta_macro_f1` > 0 means overall class balance improved.

**How to interpret the results:**
- If the fairness model has **similar accuracy** but **better fairness** (shown before),
  then the trade-off is good.
- A small drop in accuracy can be acceptable if fairness improves clearly.
- Macro F1 is important here because it shows performance on **all classes**, not only the largest ones.

**Why this is important for fairness testing:**
Fairness methods should not only reduce group differences,  
they should also keep **overall model quality** at a reasonable level.

**References:**
- Hardt et al. (2016). *Equality of Opportunity in Supervised Learning*.  
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*.  
- Mitchell et al. (2019). *Model Cards for Model Reporting*.


In [33]:
from sklearn.metrics import accuracy_score, f1_score

# =================================================
# ACCURACY & PERFORMANCE COMPARISON
# =================================================

acc_base = accuracy_score(y_true, pred_base)
acc_fair = accuracy_score(y_true, pred_fair)

f1_base = f1_score(y_true, pred_base, average="macro")
f1_fair = f1_score(y_true, pred_fair, average="macro")

perf = pd.DataFrame({
    "model": ["baseline", "fair"],
    "accuracy": [acc_base, acc_fair],
    "macro_f1": [f1_base, f1_fair],
})

perf["delta_accuracy"] = perf["accuracy"] - perf.loc[0, "accuracy"]
perf["delta_macro_f1"] = perf["macro_f1"] - perf.loc[0, "macro_f1"]

print("\n=== Overall performance comparison ===")
display(perf)



=== Overall performance comparison ===


,model,accuracy,macro_f1,delta_accuracy,delta_macro_f1
0,baseline,0.596716,0.219275,0.000000,0.000000
1,fair,0.316059,0.217693,-0.280657,-0.001582
